# NN Dataset Generation for TAG Authentication

**Objective**: Generate 100k+ synthetic samples for neural network training
- **Features**: Correlator output, channel estimation, SNR, noise statistics
- **Labels**: Authentic (1) vs Fraudulent (0)
- **Test conditions**: Multiple SNR levels (8-12 dB) and TAG lengths (512-1024)
- **Output**: Train/Val/Test split (80/10/10) saved to HDF5

**References**: 
- Braca et al. (2022) - Statistical Hypothesis Testing with ML
- Existing Monte Carlo simulations

**Date**: March 17, 2026

In [ ]:
# ==============================================================================
# 1. IMPORTS & SETUP
# ==============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc, erfcinv
from scipy import stats
import h5py
import os
from tqdm import tqdm
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")
print(f"Python version: {np.__version__}")
print(f"NumPy version: {np.__version__}")

In [ ]:
# ==============================================================================
# 2. CORE FUNCTIONS FOR TAG GENERATION & SIMULATION
# ==============================================================================

def modulator_bpsk(L, seed=None):
    """Generate BPSK modulated sequence (+1, -1)
    
    Args:
        L: Length of sequence
        seed: Random seed (optional)
    
    Returns:
        s: BPSK modulated sequence of length L
    """
    if seed is not None:
        np.random.seed(seed)
    bits = np.random.randint(0, 2, L)
    s = 2 * bits - 1  # Convert to +1/-1
    return s

def tent_map(x, beta=1e-6):
    """Tent map chaotic function
    
    Args:
        x: Input value in [0, 1]
        beta: Parameter (default 1e-6)
    
    Returns:
        output: Tent map result
    """
    return 1 - 2 * np.abs(x - 0.5) - beta

def quadratic_map(x, K):
    """Quadratic chaotic map
    
    Args:
        x: Input value
        K: Key size in bits
    
    Returns:
        M: (6*x**2 + x + 1) mod 2^K
    """
    return (6 * x**2 + x + 1) % (2**K)

def generate_tag(msg, key, L, K=512):
    """Generate chaotic TAG using tent map and quadratic map
    
    Args:
        msg: BPSK message (+1/-1)
        key: Cryptographic key (bit sequence)
        L: Length of TAG to generate
        K: Key size in bits (default 512)
    
    Returns:
        tag: Chaotic TAG of length L with expected energy E[tag**2] ≈ 1/3
    """
    # XOR message bits with key -> seed for iteration
    seed_value = np.sum(msg * key[:len(msg)]) % 1
    
    # Initialize chaotic orbit
    x = seed_value
    orbit = [x]
    
    # Generate orbit (discard first K-L points)
    for _ in range(K):
        x = tent_map(x)
        if len(orbit) > K - L:
            orbit.append(x)
    
    # Extract TAG from orbit
    tag = np.array(orbit[-(L):])
    
    # Normalize to have E[tag**2] ≈ 1/3
    tag = tag / np.sqrt(3 * np.var(tag))
    
    return tag

def rayleigh_channel(length, sigma_h=1/np.sqrt(2)):
    """Generate Rayleigh fading channel coefficient
    
    Args:
        length: Length of channel sequence
        sigma_h: Standard deviation (default 1/sqrt(2))
    
    Returns:
        h: Rayleigh fading coefficients
    """
    real_part = np.random.normal(0, sigma_h, length)
    imag_part = np.random.normal(0, sigma_h, length)
    h = np.sqrt(real_part**2 + imag_part**2)
    return h

def awgn_channel(signal, snr_db):
    """Add AWGN to signal
    
    Args:
        signal: Input signal
        snr_db: Signal-to-noise ratio in dB
    
    Returns:
        y_received: Noisy signal
    """
    snr_linear = 10**(snr_db / 10)
    signal_power = np.mean(np.abs(signal)**2)
    noise_power = signal_power / snr_linear
    noise = np.random.normal(0, np.sqrt(noise_power), len(signal))
    return signal + noise

print("Core functions defined successfully!")

In [ ]:
# ==============================================================================
# 3. DATA GENERATION PIPELINE
# ==============================================================================

def generate_training_pair(snr_db, L, K=512, is_authentic=True):
    """Generate ONE training pair (features, label)
    
    Args:
        snr_db: Signal-to-noise ratio in dB
        L: Length of TAG
        K: Key size (default 512)
        is_authentic: True=authentic (H1), False=fraudulent (H0)
    
    Returns:
        features: dict with keys [correlator_out, h_magnitude, snr_local, energy]
        label: 1 if authentic, 0 if fraudulent
    """
    
    # Parameters
    rho_s = np.sqrt(0.985)  # Message power
    rho_t = 0.124           # TAG power
    
    # Step 1: Generate BPSK message
    msg = modulator_bpsk(L)
    
    # Step 2: Generate/receive TAGs based on authenticity
    if is_authentic:
        # H1: Legitimate TAG + message
        key = np.random.randint(0, 2, K)
        tag = generate_tag(msg, key, L, K)
        
        # Channel: Rayleigh fading
        h = rayleigh_channel(1)[0]
        
        # Transmitted signal: s = rho_s * msg + rho_t * tag
        transmitted = rho_s * msg + rho_t * tag
        
        # Received signal: y = h * transmitted + noise
        received = h * transmitted
        received = awgn_channel(received, snr_db)
        
        # For detection: correlate with legitimate TAG
        tag_ref = tag  # We know the legitimate TAG
        
    else:
        # H0: Fraudulent TAG (random) + message
        msg_fake = modulator_bpsk(L)  # Different message
        key_fake = np.random.randint(0, 2, K)
        tag_fake = generate_tag(msg_fake, key_fake, L, K)
        
        # Channel: Different Rayleigh realization
        h_fake = rayleigh_channel(1)[0]
        
        # Transmitted signal (fraudulent)
        transmitted = rho_s * msg_fake + rho_t * tag_fake
        received = h_fake * transmitted
        received = awgn_channel(received, snr_db)
        
        # For detection: correlate with OUR legitimate TAG
        # (which doesn't match the fraudulent one)
        key = np.random.randint(0, 2, K)
        tag_ref = generate_tag(msg, key, L, K)
        h = h_fake  # Store for feature extraction
    
    # Step 3: Extract features
    # Feature 1: Correlator output (main statistic)
    y_minus_msg = (received / h - rho_s * msg) / rho_t  # Estimate received TAG
    correlator_out = np.abs(np.sum(y_minus_msg * tag_ref))
    
    # Feature 2: Channel estimate (magnitude)
    h_estimate = np.mean(np.abs(received))
    
    # Feature 3: Local SNR estimate
    signal_power = np.mean(np.abs(rho_s * msg)**2)
    noise_power = np.mean(np.abs(received - h * (rho_s * msg + rho_t * tag))**2) if is_authentic else np.mean(np.var(received))
    snr_local = 10 * np.log10((signal_power + 1e-10) / (noise_power + 1e-10))
    
    # Feature 4: Energy of received signal
    energy = np.mean(np.abs(received)**2)
    
    features = {
        'correlator_out': correlator_out,
        'h_magnitude': h_estimate,
        'snr_local': snr_local,
        'energy': energy
    }
    
    label = 1 if is_authentic else 0
    
    return features, label

def generate_dataset(num_samples=100000, snr_range=(8, 12), L_range=(512, 1024)):
    """Generate large dataset for NN training
    
    Args:
        num_samples: Total number of samples to generate
        snr_range: (min_snr, max_snr) in dB
        L_range: (min_L, max_L) TAG length range
    
    Returns:
        X: Feature matrix (num_samples, 4)
        y: Labels (num_samples,)
    """
    
    X_list = []
    y_list = []
    
    print(f"Generating {num_samples} samples...")
    for i in tqdm(range(num_samples)):
        # Randomly choose parameters
        snr = np.random.uniform(snr_range[0], snr_range[1])
        L = np.random.randint(L_range[0], L_range[1] + 1)
        is_authentic = (i % 2 == 0)  # Alternate H1/H0 for balance
        
        features, label = generate_training_pair(snr, L, is_authentic=is_authentic)
        
        # Stack features as row vector
        X_row = np.array([
            features['correlator_out'],
            features['h_magnitude'],
            features['snr_local'],
            features['energy']
        ])
        
        X_list.append(X_row)
        y_list.append(label)
    
    X = np.array(X_list)
    y = np.array(y_list)
    
    return X, y

print("Data generation pipeline defined!")

In [ ]:
# ==============================================================================
# 4. GENERATE DATASET
# ==============================================================================

# Configuration
NUM_SAMPLES = 100000  # 100k samples
SNR_RANGE = (8, 12)  # dB
L_RANGE = (512, 1024)

# Generate dataset
X, y = generate_dataset(
    num_samples=NUM_SAMPLES,
    snr_range=SNR_RANGE,
    L_range=L_RANGE
)

print(f"\n✓ Dataset generated!")
print(f"  Shape X: {X.shape}")
print(f"  Shape y: {y.shape}")
print(f"  Label distribution: {np.bincount(y)}")
print(f"  Feature statistics:")
print(f"    Correlator: [μ={X[:, 0].mean():.3f}, σ={X[:, 0].std():.3f}]")
print(f"    H estimate: [μ={X[:, 1].mean():.3f}, σ={X[:, 1].std():.3f}]")
print(f"    SNR local:  [μ={X[:, 2].mean():.3f}, σ={X[:, 2].std():.3f}]")
print(f"    Energy:     [μ={X[:, 3].mean():.3f}, σ={X[:, 3].std():.3f}]")

In [ ]:
# ==============================================================================
# 5. SPLIT & NORMALIZATION
# ==============================================================================

from sklearn.preprocessing import StandardScaler

# Split: 80% train, 10% val, 10% test (stratified by label)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print(f"Split completed:")
print(f"  Train: {X_train.shape} ({100*X_train.shape[0]/len(X):.1f}%)")
print(f"  Val:   {X_val.shape} ({100*X_val.shape[0]/len(X):.1f}%)")
print(f"  Test:  {X_test.shape} ({100*X_test.shape[0]/len(X):.1f}%)")
print(f"  Train labels: {np.bincount(y_train)}")
print(f"  Val labels:   {np.bincount(y_val)}")
print(f"  Test labels:  {np.bincount(y_test)}")

# Normalize features (fit scaler on train data)
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_val_norm = scaler.transform(X_val)
X_test_norm = scaler.transform(X_test)

print(f"\n✓ Normalization completed")
print(f"  Scaler mean: {scaler.mean_}")
print(f"  Scaler std:  {scaler.scale_}")

In [ ]:
# ==============================================================================
# 6. SAVE TO HDF5
# ==============================================================================

output_path = "dataset_nn_100k.h5"

with h5py.File(output_path, 'w') as f:
    # Create datasets
    f.create_dataset('X_train', data=X_train_norm, compression='gzip')
    f.create_dataset('y_train', data=y_train, compression='gzip')
    
    f.create_dataset('X_val', data=X_val_norm, compression='gzip')
    f.create_dataset('y_val', data=y_val, compression='gzip')
    
    f.create_dataset('X_test', data=X_test_norm, compression='gzip')
    f.create_dataset('y_test', data=y_test, compression='gzip')
    
    # Store scaler parameters
    f.create_dataset('scaler_mean', data=scaler.mean_)
    f.create_dataset('scaler_std', data=scaler.scale_)
    
    # Store metadata
    f.attrs['num_samples'] = NUM_SAMPLES
    f.attrs['snr_range'] = SNR_RANGE
    f.attrs['L_range'] = L_RANGE
    f.attrs['feature_names'] = ['correlator_out', 'h_magnitude', 'snr_local', 'energy']
    f.attrs['split_ratio'] = [0.8, 0.1, 0.1]

print(f"✓ Dataset saved to '{output_path}'")
print(f"  File size: {os.path.getsize(output_path) / 1024 / 1024:.2f} MB")

# Verify
with h5py.File(output_path, 'r') as f:
    print(f"\nVerification:")
    for key in f.keys():
        print(f"  {key}: {f[key].shape}")

In [ ]:
# ==============================================================================
# 7. EXPLORATORY DATA ANALYSIS
# ==============================================================================

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# Plot 1: Correlator output distribution
axes[0, 0].hist(X_train_norm[y_train==0, 0], alpha=0.5, label='Fraudulent (H0)', bins=40)
axes[0, 0].hist(X_train_norm[y_train==1, 0], alpha=0.5, label='Authentic (H1)', bins=40)
axes[0, 0].set_xlabel('Correlator output (normalized)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].legend()
axes[0, 0].set_title('Feature 0: Correlator Output')

# Plot 2: H estimate distribution
axes[0, 1].hist(X_train_norm[y_train==0, 1], alpha=0.5, label='H0', bins=40)
axes[0, 1].hist(X_train_norm[y_train==1, 1], alpha=0.5, label='H1', bins=40)
axes[0, 1].set_xlabel('H estimate (normalized)')
axes[0, 1].set_ylabel('Count')
axes[0, 1].legend()
axes[0, 1].set_title('Feature 1: Channel Estimation')

# Plot 3: SNR distribution
axes[0, 2].hist(X_train_norm[y_train==0, 2], alpha=0.5, label='H0', bins=40)
axes[0, 2].hist(X_train_norm[y_train==1, 2], alpha=0.5, label='H1', bins=40)
axes[0, 2].set_xlabel('SNR local (normalized)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].legend()
axes[0, 2].set_title('Feature 2: Local SNR')

# Plot 4: Energy distribution
axes[1, 0].hist(X_train_norm[y_train==0, 3], alpha=0.5, label='H0', bins=40)
axes[1, 0].hist(X_train_norm[y_train==1, 3], alpha=0.5, label='H1', bins=40)
axes[1, 0].set_xlabel('Energy (normalized)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].legend()
axes[1, 0].set_title('Feature 3: Signal Energy')

# Plot 5: ROC-like scatter (Correlator vs Energy)
axes[1, 1].scatter(X_train_norm[y_train==0, 0], X_train_norm[y_train==0, 3], 
                    alpha=0.3, s=10, label='H0', c='red')
axes[1, 1].scatter(X_train_norm[y_train==1, 0], X_train_norm[y_train==1, 3], 
                    alpha=0.3, s=10, label='H1', c='blue')
axes[1, 1].set_xlabel('Correlator (normalized)')
axes[1, 1].set_ylabel('Energy (normalized)')
axes[1, 1].legend()
axes[1, 1].set_title('Feature Space (Correlator vs Energy)')

# Plot 6: Label distribution by split
axes[1, 2].bar(['Train', 'Val', 'Test'], 
               [np.sum(y_train==1), np.sum(y_val==1), np.sum(y_test==1)],
               label='Authentic (1)', alpha=0.8)
axes[1, 2].bar(['Train', 'Val', 'Test'], 
               [np.sum(y_train==0), np.sum(y_val==0), np.sum(y_test==0)],
               bottom=[np.sum(y_train==1), np.sum(y_val==1), np.sum(y_test==1)],
               label='Fraudulent (0)', alpha=0.8)
axes[1, 2].set_ylabel('Number of samples')
axes[1, 2].set_title('Label Distribution by Split')
axes[1, 2].legend()

plt.tight_layout()
plt.savefig('dataset_exploration.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Exploratory plots saved to 'dataset_exploration.png'")

## Summary

✅ **Dataset Generation Complete!**

### Statistics
- **Total samples**: 100,000
- **Features**: 4 (correlator_out, h_magnitude, snr_local, energy)
- **Classes**: Binary (Authentic=1, Fraudulent=0)
- **Split**: Train=80k (80%), Val=10k (10%), Test=10k (10%)
- **SNR range**: 8-12 dB
- **TAG length range**: 512-1024 symbols

### Feature Description
| Feature | Description | Purpose |
|---------|-------------|---------|
| `correlator_out` | Correlation of received signal with legitimate TAG | Main detection statistic |
| `h_magnitude` | Estimated channel magnitude | Channel estimation |
| `snr_local` | Local SNR estimate | Noise level adaptation |
| `energy` | Energy of received signal | Signal amplitude adaptation |

### Output Files
- `dataset_nn_100k.h5` - Main dataset with normalized features
- `dataset_exploration.png` - Feature distribution visualizations

### Next Steps
1. **NN_02_DNN_Correlator.ipynb** - Implement Correlator+DNN (Braca et al. 2022)
2. **NN_03_CNN_SignalProcessing.ipynb** - Implement CNN 1D ("Binary Case using Deep Learning")
3. **NN_04_LSTM_Rayleigh.ipynb** - Implement LSTM for Rayleigh fading
4. **NN_05_Ensemble_Hybrid.ipynb** - Combine models for robustness
5. **NN_06_Comparison_vs_Baseline.ipynb** - Validate against Monte Carlo baseline

### References
- **[1]** Braca, P., et al. (2022). "Statistical Hypothesis Testing Based on Machine Learning: Large Deviations Analysis." IEEE Open Journal of Signal Processing, 3, 464-495.
- **[2]** "Binary Case using Deep Learning" (project paper)